# Beginner Workshop: Single-Cell RNA-seq Analysis with Scanpy
## PBMC 3k Tutorial — Basics & Visualization

Welcome to this hands-on workshop on single-cell RNA sequencing (scRNA-seq) analysis using **Scanpy** and other tools from the [scverse](https://scverse.org/) ecosystem.

### Reference Tutorials
This workshop notebook is inspired by the following official Scanpy tutorials:
- [Preprocessing and clustering 3k PBMCs (legacy workflow)](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/clustering-2017.html)
- [Preprocessing and clustering (modern workflow)](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/clustering.html)
- [Integrating data using ingest](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/integrating-data-using-ingest.html)
- [Core plotting functions](https://scanpy.readthedocs.io/en/1.10.x/tutorials/plotting/core.html)
- [Advanced plotting](https://scanpy.readthedocs.io/en/1.10.x/tutorials/plotting/advanced.html)

### Workshop Overview
| Section | Topic |
|---------|-------|
| 0 | Setup and imports |
| 1 | The AnnData data structure |
| 2 | Loading PBMC data |
| 3 | Exploring raw data |
| 4 | Quality control (QC) and filtering |
| 5 | Doublet detection |
| 6 | Normalization, HVGs, and scaling |
| 7 | Cell cycle scoring |
| 8 | Principal Component Analysis (PCA) |
| 9 | Clustering with Leiden |
| 10 | Embedding visualization (UMAP, t-SNE) |
| 11 | Visualization gallery |
| 12 | Marker gene detection |
| 13 | Cell type annotation |
| 14 | Data integration with ingest |
| 15 | Save results and workshop exercises |

### Dataset
We use the [PBMC 3k](https://support.10xgenomics.com/single-cell-gene-expression/datasets/1.1.0/pbmc3k) dataset — 2,700 peripheral blood mononuclear cells from a healthy donor sequenced on the 10x Chromium v1 platform. This is the standard Scanpy tutorial dataset.

> **Prerequisites:** basic Python knowledge. No prior scRNA-seq experience required.


## 0. Setup

Install the required packages if they are not already available:
```bash
pip install scanpy anndata matplotlib seaborn scrublet session-info
```

The key packages we use:
- **scanpy** — single-cell analysis in Python
- **anndata** — annotated data matrix (the core data structure)
- **matplotlib / seaborn** — plotting
- **scrublet** — doublet detection


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad

# Reproducibility
np.random.seed(42)

# Scanpy global settings
sc.settings.verbosity = 3          # print progress messages (0=errors only, 3=hints)
sc.settings.n_jobs = 1             # single CPU; increase for large datasets
sc.set_figure_params(dpi=100, facecolor='white', figsize=(5, 4))

print('scanpy version:', sc.__version__)
print('anndata version:', ad.__version__)


## 1. The AnnData Data Structure

Almost everything in Scanpy revolves around an **AnnData** object. Think of it as a smart container that holds the expression matrix together with all associated metadata.

```
              genes (var)
          ┌────────────────────────┐
   cells  │                        │
   (obs)  │    X  (expression      │
          │       matrix)          │
          └────────────────────────┘
```

| Slot | What it holds |
|------|---------------|
| `adata.X` | count / expression matrix (cells × genes) |
| `adata.obs` | cell metadata (e.g., QC metrics, cluster labels) |
| `adata.var` | gene metadata (e.g., HVG flags, mean expression) |
| `adata.obsm` | cell-level embeddings (e.g., PCA, UMAP) |
| `adata.obsp` | pairwise cell relationships (e.g., kNN graph) |
| `adata.uns` | unstructured metadata (colors, neighbor params, etc.) |
| `adata.layers` | additional expression matrices (raw counts, batch-corrected) |
| `adata.raw` | frozen snapshot before HVG-subsetting |

> **Workshop question:** Why is it useful to keep all this information in one container?


## 2. Loading PBMC Data

### Option A — Built-in dataset (recommended for this workshop)
Scanpy ships with the PBMC 3k dataset via `sc.datasets.pbmc3k()`.

### Option B — Load from 10x CellRanger output
In a real project you would load your own data from a CellRanger output directory:
```python
# Download and unpack the data first:
# mkdir -p data && cd data
# curl https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz \
#      -o pbmc3k_filtered_gene_bc_matrices.tar.gz
# tar -xzf pbmc3k_filtered_gene_bc_matrices.tar.gz

adata = sc.read_10x_mtx(
    'data/filtered_gene_bc_matrices/hg19/',
    var_names='gene_symbols',
    cache=True,
)
```


In [ ]:
# Load PBMC3k using the built-in helper (downloads automatically)
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()   # ensure all gene names are unique

print('Shape:', adata.shape)    # (n_cells, n_genes)
print(adata)


In [ ]:
# Inspect the cell and gene metadata tables
print('obs (cell metadata):'); print(adata.obs.head())
print()
print('var (gene metadata):'); print(adata.var.head())


In [ ]:
# Preserve raw counts in a separate layer for later use
adata.layers['counts'] = adata.X.copy()

# Where do we save processed files?
results_file = 'pbmc3k_workshop.h5ad'


## 3. Exploring Raw Data

Before any filtering it is useful to visualise which genes dominate the total counts. The plot below shows the top 20 genes ranked by mean fraction of counts per cell.

> **Tip:** ribosomal proteins and mitochondrial genes often appear at the top.


In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)


## 4. Quality Control and Filtering

Before analysis we remove **low-quality cells** and uninformative genes.

### Key QC metrics
| Metric | What it measures | Problem indicated |
|--------|-----------------|-------------------|
| `n_genes_by_counts` | Number of detected genes | Low → empty droplet; very high → doublet |
| `total_counts` | Total UMIs (library size) | Closely correlated with gene count |
| `pct_counts_mt` | % of counts from mt genes | High → damaged/dying cell |
| `pct_counts_ribo` | % of counts from ribo genes | Very high may indicate stress |

Mitochondrial gene names start with **`MT-`** (human) or **`mt-`** (mouse).


In [ ]:
# Flag mitochondrial and ribosomal genes
adata.var['mt']   = adata.var_names.str.startswith('MT-')
adata.var['ribo'] = adata.var_names.str.startswith(('RPS', 'RPL'))

# Compute per-cell QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt', 'ribo'],
    percent_top=None,
    log1p=False,
    inplace=True,
)

print('QC columns added to adata.obs:')
print([c for c in adata.obs.columns])


In [ ]:
# Violin plots — overview of three key QC metrics
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo'],
    jitter=0.4,
    multi_panel=True,
)


In [ ]:
# Scatter plots — visually identify outlier cells
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', ax=axes[0], show=False)
axes[0].axhline(5, color='red', linestyle='--', label='5% MT threshold')
axes[0].legend()

sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=axes[1], show=False)
axes[1].axhline(2500, color='red', linestyle='--', label='2500 gene threshold')
axes[1].legend()

plt.tight_layout()
plt.show()


### 4.1 Apply Filters

Standard PBMC 3k tutorial thresholds:
- ≥ 200 genes (removes empty droplets)
- < 2500 genes (removes likely doublets)
- < 5% mitochondrial reads (removes damaged cells)
- Expressed in ≥ 3 cells (removes uninformative genes)

> **Workshop question:** How would you choose appropriate thresholds for a *new* dataset?


In [ ]:
print(f'Before filtering: {adata.n_obs} cells, {adata.n_vars} genes')

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

adata = adata[adata.obs.n_genes_by_counts < 2500, :].copy()
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()

print(f'After filtering:  {adata.n_obs} cells, {adata.n_vars} genes')


## 5. Doublet Detection with Scrublet

**Doublets** are droplets that captured two cells. They can create spurious clusters and confound downstream analysis.

[Scrublet](https://github.com/swolock/scrublet) simulates synthetic doublets by combining pairs of real cells and gives each observed cell a **doublet score** reflecting its similarity to those simulated doublets.

> **Note:** Scrublet expects **raw counts**. We use the `counts` layer we saved earlier.


In [ ]:
try:
    import scrublet as scr

    scrub = scr.Scrublet(adata.layers['counts'])
    doublet_scores, predicted_doublets = scrub.scrub_doublets()

    adata.obs['doublet_score'] = doublet_scores
    adata.obs['predicted_doublet'] = predicted_doublets

    scrub.plot_histogram()
    plt.show()

    n_d = predicted_doublets.sum()
    print(f'Predicted doublets: {n_d} / {adata.n_obs} ({100*n_d/adata.n_obs:.1f}%)')

    adata = adata[~adata.obs['predicted_doublet']].copy()
    print(f'After doublet removal: {adata.n_obs} cells')

except ImportError:
    print('scrublet not installed — skipping. Install with: pip install scrublet')


## 6. Normalization, HVG Selection, and Scaling

### 6.1 Normalization

Cells are sequenced to different depths (some deeper than others). We equalise them by:
1. **Total-count normalization** — scale each cell to 10,000 UMIs
2. **log1p transformation** — `log(x + 1)`, compresses dynamic range and stabilises variance


In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Save a snapshot of normalised, log-transformed data before HVG subsetting.
# adata.raw retains all genes for downstream visualisation.
adata.raw = adata

print('Normalization complete. adata.raw holds the full gene expression matrix.')


### 6.2 Highly Variable Genes (HVGs)

Most genes vary little across cells — they add noise rather than signal. We select the **highly variable genes** (those that vary more than expected by chance) to focus downstream analyses.

Scanpy's default method (Seurat v1): genes are binned by mean expression; within each bin the normalised dispersion (variance/mean) is computed; genes with dispersion above a threshold are selected.


In [ ]:
sc.pp.highly_variable_genes(
    adata,
    min_mean=0.0125,
    max_mean=3,
    min_disp=0.5,
)

print(f'HVGs: {adata.var.highly_variable.sum()} / {adata.n_vars}')
sc.pl.highly_variable_genes(adata)


### 6.3 Regress Out Confounders and Scale

We **regress out** total UMI counts and mitochondrial fraction so they do not dominate PCA. Then we **scale** each gene to unit variance (z-score) so all genes contribute equally.

> **Note:** We first subset to HVGs (keeping `.raw` for full-gene visualisation), then regress and scale.


In [ ]:
# Subset to HVGs only; adata.raw still holds all genes
adata = adata[:, adata.var.highly_variable].copy()

# Regress out technical confounders
sc.pp.regress_out(adata, ['total_counts', 'pct_counts_mt'])

# Scale to unit variance; clip extreme outliers at 10 SDs
sc.pp.scale(adata, max_value=10)

print('Preprocessing done. Shape after HVG subset:', adata.shape)


## 7. Cell Cycle Scoring

Cell-cycle-driven variation can confound clustering. We score each cell for its **S-phase** and **G2/M-phase** activity using the gene lists from Tirosh et al. 2016.

> **Note:** For this PBMC dataset, cell-cycle effects are modest. They are more prominent in actively cycling datasets (tumour, stem cells, etc.).


In [ ]:
# Tirosh et al. 2016 cell cycle gene lists
s_genes = [
    'MCM5', 'PCNA', 'TYMS', 'FEN1', 'MCM2', 'MCM4', 'RRM1', 'UNG',
    'GINS2', 'MCM6', 'CDCA7', 'DTL', 'PRIM1', 'UHRF1', 'MLF1IP',
    'HELLS', 'RFC2', 'RPA2', 'NASP', 'RAD51AP1', 'GMNN', 'WDR76',
    'SLBP', 'CCNE2', 'UBR7', 'POLD3', 'MSH2', 'ATAD2', 'RAD51',
    'RRM2', 'CDC45', 'CDC6', 'EXO1', 'TIPIN', 'DSCC1', 'BLM',
    'CASP8AP2', 'USP1', 'CLSPN', 'POLA1', 'CHAF1B', 'BRIP1', 'E2F8',
]

g2m_genes = [
    'HMGB2', 'CDK1', 'NUSAP1', 'UBE2C', 'BIRC5', 'TPX2', 'TOP2A',
    'NDC80', 'CKS2', 'NUF2', 'CKS1B', 'MKI67', 'TMPO', 'CENPF',
    'TACC3', 'FAM64A', 'SMC4', 'CCNB2', 'CKAP2L', 'CKAP2', 'AURKB',
    'BUB1', 'KIF11', 'ANP32E', 'TUBB4B', 'GTSE1', 'KIF20B', 'HJURP',
    'CDCA3', 'HN1', 'CDC20', 'TTK', 'CDC25C', 'KIF2C', 'RANGAP1',
    'NCAPD2', 'DLGAP5', 'CDCA2', 'CDCA8', 'ECT2', 'KIF23', 'HMMR',
    'AURKA', 'PSRC1', 'ANLN', 'LBR', 'CKAP5', 'CENPE', 'CTCF',
    'NEK2', 'G2E3', 'GAS2L3', 'CBX5', 'CENPA',
]

# Filter to genes present in our HVG-subset data
s_use   = [g for g in s_genes   if g in adata.var_names]
g2m_use = [g for g in g2m_genes if g in adata.var_names]

if s_use and g2m_use:
    sc.tl.score_genes_cell_cycle(adata, s_genes=s_use, g2m_genes=g2m_use)
    print('Cell cycle phase distribution:')
    print(adata.obs['phase'].value_counts())
else:
    print('Not enough cell cycle genes in HVG subset — skipping.')


## 8. Principal Component Analysis (PCA)

PCA projects the ~2 000-gene HVG space into a smaller set of **principal components (PCs)** that capture the most variation.

### Steps
1. Compute PCs
2. Inspect the **elbow plot** to choose how many PCs to use
3. Visualise cells in PC space

> **Tip:** The elbow plot shows variance explained per PC. The 'elbow' — where the curve flattens — is a common heuristic for choosing the number of PCs.


In [ ]:
sc.tl.pca(adata, n_comps=50)  # svd_solver='auto' is the default and works well for typical HVG subset sizes

# Elbow plot — variance explained per PC
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)


In [ ]:
# Scatter plot in PC space, coloured by QC metrics
sc.pl.pca(
    adata,
    color=['total_counts', 'pct_counts_mt'],
    ncols=2,
)


In [ ]:
# Colour by cell cycle phase if scored
if 'phase' in adata.obs.columns:
    sc.pl.pca(adata, color='phase')


> **Workshop question:** How many PCs capture most of the meaningful biological variation in the elbow plot?


## 9. Clustering with the Leiden Algorithm

Graph-based clustering in two steps:
1. **Build a k-nearest-neighbour (kNN) graph** in PC space
2. **Detect communities** using the Leiden algorithm

The `resolution` parameter controls granularity: higher → more clusters.

> **Workshop note:** We compute UMAP here too so we can visualise clusters immediately below.


In [ ]:
# Build kNN graph (40 PCs, 10 neighbours)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# Compute UMAP
sc.tl.umap(adata)

# Leiden clustering at three resolutions
for res in [0.3, 0.5, 1.0]:
    key = f'leiden_res{res}'
    sc.tl.leiden(adata, resolution=res, key_added=key)
    print(f'Leiden res={res}: {adata.obs[key].nunique()} clusters')


## 10. Embedding Visualisations (UMAP & t-SNE)

### 10.1 UMAP

UMAP (Uniform Manifold Approximation and Projection) projects cells into 2D while preserving local neighbourhood structure. It is the most widely used embedding in single-cell analysis.

> **Important:** UMAP distances are *not* globally meaningful — do not over-interpret the positions of distant clusters.


In [ ]:
# Clustering at three resolutions side by side
sc.pl.umap(
    adata,
    color=['leiden_res0.3', 'leiden_res0.5', 'leiden_res1.0'],
    ncols=3,
    legend_loc='on data',
    title=['Leiden (res=0.3)', 'Leiden (res=0.5)', 'Leiden (res=1.0)'],
)


In [ ]:
# QC metrics on UMAP — check if any cluster is low quality
sc.pl.umap(
    adata,
    color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'],
    ncols=3,
    title=['Total UMI counts', 'Genes per cell', '% MT reads'],
)


In [ ]:
# Canonical PBMC marker genes on UMAP
# (using .raw so we show all genes, not just HVGs)
marker_genes_preview = {
    'CD3D':  'T cells',
    'CD19':  'B cells',
    'CD14':  'Monocytes',
    'GNLY':  'NK cells',
    'FCER1A':'Dendritic cells',
    'PPBP':  'Megakaryocytes',
}
genes_present = [g for g in marker_genes_preview if g in adata.raw.var_names]
sc.pl.umap(
    adata,
    color=genes_present,
    use_raw=True,
    ncols=3,
    title=[f'{g} ({marker_genes_preview[g]})' for g in genes_present],
)


### 10.2 t-SNE

t-SNE is an alternative 2D embedding that tends to produce tighter clusters but is slower and less scalable than UMAP.


In [ ]:
sc.tl.tsne(adata, use_rep='X_pca', learning_rate='auto')
sc.pl.tsne(adata, color='leiden_res0.5', legend_loc='on data')


## 11. Visualization Gallery

Scanpy provides a rich set of plotting functions for exploring gene expression patterns. We showcase the main ones here.

We use clusters at **res=0.5** and a panel of canonical PBMC marker genes.


In [ ]:
# Canonical PBMC marker gene panels (grouped by cell type)
pbmc_markers = {
    'CD4 T': ['IL7R', 'CCR7', 'S100A4'],
    'CD8 T': ['CD8A', 'CD8B'],
    'B':     ['CD19', 'MS4A1', 'CD79A'],
    'NK':    ['GNLY', 'NKG7', 'GZMB'],
    'Mono':  ['CD14', 'LYZ', 'CST3'],
    'DC':    ['FCER1A', 'CST3'],
    'Plt':   ['PPBP'],
}

# Flatten to unique genes that are actually in the dataset
all_markers = list(dict.fromkeys(
    g for genes in pbmc_markers.values() for g in genes
    if g in adata.raw.var_names
))
print('Marker genes available:', all_markers)


### 11.1 Dot Plot

Shows the **fraction of cells** expressing a gene (dot size) and the **mean expression level** (colour) per group.


In [ ]:
sc.pl.dotplot(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    dendrogram=True,
    use_raw=True,
    standard_scale='var',
    title='Marker gene expression per cluster',
)


### 11.2 Violin Plot

Shows the **distribution of expression** per cluster for individual genes.


In [ ]:
sc.pl.violin(
    adata,
    keys=all_markers[:6],
    groupby='leiden_res0.5',
    rotation=45,
    use_raw=True,
)


### 11.3 Heatmap

Shows mean expression per cluster as a colour grid. Useful for a quick overview of many markers at once.


In [ ]:
sc.pl.heatmap(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
    cmap='viridis',
)


### 11.4 Matrix Plot

Similar to a heatmap but averages expression within groups and uses a diverging colour scale.


In [ ]:
sc.pl.matrixplot(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
    cmap='RdBu_r',
)


### 11.5 Stacked Violin Plot

Combines violin plots for multiple genes into a compact stacked layout.


In [ ]:
sc.pl.stacked_violin(
    adata,
    var_names=pbmc_markers,
    groupby='leiden_res0.5',
    use_raw=True,
    dendrogram=True,
)


### 11.6 Tracks Plot

Shows expression as horizontal bars for each cell, sorted by cluster. Excellent for seeing within-cluster heterogeneity.


In [ ]:
sc.pl.tracksplot(
    adata,
    var_names=all_markers[:8],
    groupby='leiden_res0.5',
    use_raw=True,
)


## 12. Marker Gene Detection

To understand what each cluster represents we perform **differential expression** — finding genes that are highly expressed in one cluster compared to all others (one-vs-rest).

### Available tests
| Method | Notes |
|--------|-------|
| `wilcoxon` | Non-parametric; robust; **recommended** |
| `t-test` | Fast; assumes normality |
| `logreg` | Logistic regression; useful with many clusters |


In [ ]:
# Rank genes using Wilcoxon test; use .raw (all genes)
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden_res0.5',
    method='wilcoxon',
    use_raw=True,
    n_genes=25,
)

# Overview panel: top 15 markers per cluster
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)


In [ ]:
# Table of top markers
marker_df = sc.get.rank_genes_groups_df(adata, group=None)
print(marker_df.head(20))


In [ ]:
# Dot plot: top 3 marker genes per cluster
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes=3,
    use_raw=True,
    standard_scale='var',
)


In [ ]:
# Violin: expression of top markers for clusters 0, 1, 2
sc.pl.rank_genes_groups_violin(
    adata,
    groups=['0', '1', '2'],
    n_genes=5,
    use_raw=True,
)


## 13. Cell Type Annotation

We annotate each cluster by comparing its marker genes with published PBMC signatures.

### Canonical PBMC marker genes
| Cell type | Key markers |
|-----------|------------|
| CD4+ T cells | IL7R, CCR7, S100A4 |
| CD8+ T cells | CD8A, CD8B |
| B cells | CD19, MS4A1, CD79A |
| NK cells | GNLY, NKG7, GZMB |
| CD14+ Monocytes | CD14, LYZ, CST3 |
| FCGR3A+ Monocytes | FCGR3A, MS4A7 |
| Dendritic cells | FCER1A, CST3 |
| Megakaryocytes | PPBP |

> **Workshop exercise:** Compare your cluster markers (Section 12) with the table above to determine each cluster's identity.


In [ ]:
# Example annotation mapping
# NOTE: cluster numbers vary between runs — adjust to match your own results!
cluster_to_celltype = {
    '0': 'CD4 T',
    '1': 'CD14+ Mono',
    '2': 'CD4 T',
    '3': 'NK / CD8 T',
    '4': 'B',
    '5': 'CD8 T',
    '6': 'FCGR3A+ Mono',
    '7': 'NK',
    '8': 'DC',
    '9': 'Platelet',
}

adata.obs['cell_type'] = (
    adata.obs['leiden_res0.5']
         .map(cluster_to_celltype)
         .fillna('Unknown')
         .astype('category')
)

sc.pl.umap(
    adata,
    color='cell_type',
    legend_loc='on data',
    title='Annotated PBMC cell types',
    frameon=False,
)


In [ ]:
# Dot plot with annotated cell types
sc.pl.dotplot(
    adata,
    var_names=pbmc_markers,
    groupby='cell_type',
    use_raw=True,
    standard_scale='var',
    dendrogram=True,
)


In [ ]:
# Bar chart of cell type proportions
cell_counts = adata.obs['cell_type'].value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
cell_counts.plot.bar(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Cell type')
ax.set_ylabel('Number of cells')
ax.set_title('Cell type composition — PBMC 3k')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 14. Data Integration with `sc.tl.ingest`

*Reference: [Integrating data using ingest and BBKNN](https://scanpy.readthedocs.io/en/1.10.x/tutorials/basics/integrating-data-using-ingest.html)*

When you have a well-annotated **reference dataset** and a new **query dataset**, you can transfer labels from the reference to the query using `sc.tl.ingest`.

### How it works
1. Fit a PCA + kNN model on the reference
2. Project query cells into the reference's PCA space
3. Assign each query cell the label of its nearest neighbour in the reference

**Advantages over batch correction approaches (Harmony, BBKNN, etc.)**:
- Transparent and fast
- Solves the label-transfer problem directly
- Maintains the reference embedding structure

> **Note:** This asymmetric approach (*ingesting* annotations from reference → query) is different from jointly integrating datasets.


In [ ]:
# Load the pre-processed PBMC3k as reference
# (Scanpy's built-in processed version already has cell type labels)
adata_ref = sc.datasets.pbmc3k_processed()

# Load the PBMC 68k reduced dataset as query
adata_query = sc.datasets.pbmc68k_reduced()

print('Reference:', adata_ref.shape, '| Labels:', adata_ref.obs['louvain'].unique().tolist())
print('Query:    ', adata_query.shape)


In [ ]:
# ingest requires datasets to share the same variable names
var_names = adata_ref.var_names.intersection(adata_query.var_names)
adata_ref   = adata_ref[:, var_names].copy()
adata_query = adata_query[:, var_names].copy()

print('Shared genes:', len(var_names))


In [ ]:
# Train the PCA/kNN model on the reference
sc.pp.pca(adata_ref)
sc.pp.neighbors(adata_ref)
sc.tl.umap(adata_ref)

# Check the reference UMAP
sc.pl.umap(adata_ref, color='louvain', title='Reference PBMC3k (annotated)')


In [ ]:
# Ingest: project query cells into reference space and transfer 'louvain' labels
sc.tl.ingest(adata_query, adata_ref, obs='louvain')

# Visualise query cells projected onto the reference UMAP
sc.pl.umap(
    adata_query,
    color=['louvain'],
    title='Query PBMC68k — transferred labels',
)


In [ ]:
# Concatenate reference and query for a combined view
adata_combined = adata_ref.concatenate(
    adata_query,
    batch_categories=['3k', '68k'],
)

sc.pl.umap(
    adata_combined,
    color=['louvain', 'batch'],
    title=['Cell type (transferred)', 'Dataset batch'],
    ncols=2,
)


## 15. Save Results


In [ ]:
# Save the annotated PBMC3k object
adata.write(results_file)
print('Saved to', results_file)


## Workshop Exercises

### Beginner
1. **Change QC thresholds** — try `pct_counts_mt < 10` instead of 5. How does it affect cell numbers and clusters?
2. **Change Leiden resolution** — try `resolution=2.0`. Does the additional granularity make biological sense?
3. **Visualise a new marker gene** — look up another PBMC marker and plot it on the UMAP.

### Intermediate
4. **Ribosomal filter** — filter cells with `pct_counts_ribo > 50`. Does this change anything?
5. **Compare UMAP and t-SNE** — what structural differences do you observe?
6. **Regress out cell cycle** — add `S_score` and `G2M_score` to the `regress_out` call. Does this flatten the cycling population?

### Advanced
7. **Alternative HVG method** — use `flavor='seurat_v3'` in `sc.pp.highly_variable_genes`. Compare the selected genes.
8. **BBKNN batch correction** — install `bbknn` and run `sc.external.pp.bbknn` on the combined PBMC3k+68k object. Compare with the `ingest` result.
9. **Pseudotime** — install `scvelo` or use `sc.tl.diffmap`/`sc.tl.dpt` to compute pseudotime ordering of T cells.

### Reflection questions
- Why do we store `adata.raw` before subsetting to HVGs?
- What happens if you skip the `regress_out` step?
- Why does the number of PCs used for the neighbour graph matter?
- When would you prefer `sc.tl.ingest` over joint batch correction methods?
